In [11]:
# --- Celda 1.1: Importaciones y Carga de Librerías Offline ---
import os
import numpy as np
import pandas as pd
import glob
import ROOT
import time
import traceback
from multiprocessing import Pool, cpu_count # <-- Importamos Pool
from functools import partial

In [12]:
# Esta es la parte MÁS IMPORTANTE:
# Asegúrate de que estás corriendo este Jupyter Lab desde una terminal
# donde ANTES hiciste: source /ruta/a/auger/offline/this-auger-offline.sh

AugerOfflineRoot = os.environ.get("AUGEROFFLINEROOT")
if AugerOfflineRoot is None:
    raise EnvironmentError(
        "AUGEROFFLINEROOT no definido. "
        "Reinicia Jupyter Lab desde una terminal donde hayas "
        "hecho: "
        " 'aug_set_version offline 4.0.1-icrc23-prod1-root6' "   
        " 'source /srv/software/amd64/ubuntu/24.04/auger/offline/4.0.1-icrc23-prod1-root6/bin/this-auger-offline.sh'."
    )

print(f"AUGEROFFLINEROOT encontrado en: {AugerOfflineRoot}")

# Cargar las librerías necesarias
print("Cargando librerías de Auger Offline...")
libs_to_load = ["libRecEventKG.so"]
for lib in libs_to_load:
    lib_path = os.path.join(AugerOfflineRoot, "lib", lib)
    if not os.path.exists(lib_path):
        raise FileNotFoundError(f"No se encontró la librería: {lib_path}")
    
    # Usamos gSystem.Load que es más robusto en PyROOT
    status = ROOT.gSystem.Load(lib_path)
    if status < 0:
        raise ImportError(f"Error cargando la librería: {lib_path}")

print("Librerías cargadas correctamente. ¡Listo para trabajar! 🚀")

AUGEROFFLINEROOT encontrado en: /srv/software/amd64/ubuntu/24.04/auger/offline/4.0.1-icrc23-prod1-root6
Cargando librerías de Auger Offline...
Librerías cargadas correctamente. ¡Listo para trabajar! 🚀


In [18]:
# --- Celda 2.1: Funciones Auxiliares y Principales (con Flag de Saturación) ---

# --- getModuleList sigue igual ---
def getModuleList(counter, sim=True):
    possibleModules = range(0, 6) if sim else range(100, 116)
    modules = []
    for modId in possibleModules:
        if counter.HasModule(modId):
            modules.append(counter.GetModule(modId))
    return modules

def readADST_surface_v6(fname, is_mc_simulation=True):
    """
    Leer un archivo ADST (1 fila por MÓDULO).
    Itera sobre MDEvent.
    Guarda un flag 'is_sd_saturated' en lugar de filtrar.
    """
    
    print(f"Iniciando lectura de: {os.path.basename(fname)}")
    
    if not os.path.exists(fname):
        print(f"Advertencia: Archivo no encontrado {fname}")
        return pd.DataFrame() # Retorna DF vacío

    files = ROOT.std.vector('string')()
    files.push_back(fname)

    file1 = ROOT.RecEventFile(files)
    event = ROOT.RecEvent()
    geo = ROOT.DetectorGeometry()
    
    file1.ReadDetectorGeometry(geo)
    file1.SetBuffers(event)

    data = [] 
    event_count = 0
    start_time = time.time()

    while file1.ReadNextEvent() == ROOT.RecEventFile.eSuccess:
        event_count += 1
        if event_count % 500 == 0:
            print(f"... procesados {event_count} eventos.")

        event_id_lluvia = event.GetEventId()
        
        MCShower = event.GetGenShower()
        mc_energy = MCShower.GetEnergy()
        logE_MC = np.log10(mc_energy) if mc_energy > 0 else np.nan
        theta_MC = MCShower.GetZenith() * 180.0 / np.pi
        phi_MC = MCShower.GetAzimuth() * 180.0 / np.pi
        primary = MCShower.GetShortPrimaryName()
        
        sEvent = event.GetSDEvent()
        sShower = sEvent.GetSdRecShower()
        rec_energy = sShower.GetEnergy()
        logE_REC = np.log10(rec_energy) if rec_energy > 0 else np.nan
        theta_REC = sShower.GetZenith() * 180.0 / np.pi
        phi_REC = sShower.GetAzimuth() * 180.0 / np.pi
        
        mEvent = event.GetMDEvent()
        counterIterator = mEvent.CountersBegin()
        countersEnd = mEvent.CountersEnd()
        
        while counterIterator != countersEnd:
            
            counter = counterIterator.__deref__()
            counterId = counter.GetId()
            sdId = counter.GetSdPartnerId()

            sdStation = sEvent.GetStationById(sdId) if sEvent.HasStation(sdId) else None
            
            # --- ❗️ CAMBIO DE LÓGICA ❗️ ---
            # 1. Seguimos necesitando esto. Sin sdStation, no hay geometría.
            if sdStation is None:
                counterIterator += 1
                continue 

            # 2. ¡TU IDEA! Creamos el flag en lugar de filtrar.
            is_sd_saturated = sdStation.IsLowGainSaturated()
            # --- ❗️ FIN DEL CAMBIO ❗️ ---

            # Extraemos la geometría (ahora la extraemos siempre)
            r = sdStation.GetSPDistance()
            phi_rel = sdStation.GetAzimuthSP() - sShower.GetAzimuth()
            x_plane = r * np.cos(phi_rel)
            y_plane = r * np.sin(phi_rel)
            r_core = r
            phi_plane = np.arctan2(y_plane, x_plane) % (2 * np.pi)
            
            r_core_err = sdStation.GetSPDistanceError()
            sdSignal = sdStation.GetTotalSignal()
            sdSignal_err = sdStation.GetTotalSignalError()
            sdMuonSignal = sdStation.GetMuonSignal()

            modules = getModuleList(counter, sim=is_mc_simulation)
            for module in modules:
                
                nMuones = module.GetNumberOfEstimatedMuons()
                moduleId = module.GetId()

                if module.IsCandidate(): status = "candidate"
                elif module.IsSaturated(): status = "saturated"
                elif module.IsRejected(): status = "rejected"
                elif module.IsSilent(): status = "silent"
                else: status = "undefined"


                data.append({
                    "event_id": event_id_lluvia,
                    
                    # Info MC
                    "logE_MC": logE_MC, "theta_MC": theta_MC, "phi_MC": phi_MC, "primary": primary,
                    
                    # Info REC
                    "logE_REC": logE_REC, "theta_REC": theta_REC, "phi_REC": phi_REC,
                    
                    # Info Módulo/Counter
                    "counterId": counter.GetId(),
                    "moduleId": moduleId,
                    "nMuones": nMuones,
                    "module_status": status,      # <-- ¡NUEVO!
                    "is_sd_saturated": is_sd_saturated, # <-- ¡NUEVA COLUMNA!
                    
                    # Info de Geometría (plano de lluvia)
                    "x_plane": x_plane,
                    "y_plane": y_plane,
                    "phi_plane": phi_plane,
                    "r_core": r_core,
                    "r_core_err": r_core_err,      # <-- ¡NUEVO!
                    
                    # Info de Señal (SD)
                    "sdId": sdId,
                    "sdSignal": sdSignal,
                    "sdSignal_err": sdSignal_err,  # <-- ¡NUEVO!
                    "sdMuonSignal": sdMuonSignal   # <-- ¡NUEVO!
                })
            
            counterIterator += 1
    

    end_time = time.time()
    elapsed = end_time - start_time
    
    # --- ❗️❗️❗️ LÍNEA RESTAURADA ❗️❗️❗️ ---
    print(f"Lectura completa. Total de eventos leídos: {event_count}")
    print(f"Tiempo total de lectura: {elapsed:.2f} segundos.") # <-- ESTA FALTABA
    print(f"Total de 'MÓDULOS' (filas) extraídos: {len(data)}")
    # --- ❗️❗️❗️ FIN ❗️❗️❗️ ---

    df = pd.DataFrame(data)
    return df

In [19]:
# -----------------------------------------------------------------
# FUNCIÓN "TRABAJADORA"
# ❗️ 2. AHORA ACEPTA 'output_dir' COMO ARGUMENTO
# -----------------------------------------------------------------
def process_file_wrapper(root_fpath, output_dir):
    
    # 1. Definir rutas
    filename = os.path.basename(root_fpath)
    output_filename = filename.replace(".root", ".parquet")
    output_path = os.path.join(output_dir, output_filename)
    
    # 2. Evitar reprocesar
    if os.path.exists(output_path):
        return f"INFO: El archivo ya existe, saltando: {output_filename}"

    # 3. Imprimir estado
    print(f"► [Iniciando]: {filename}")
    
    try:
        # ----- INICIO DEL TRABAJO -----
        start_file_time = time.time()
        
        # 1. Leer el .root (¡Llama a la función de la Celda 2.1!)
        # (Asegurate de que tu Celda 2.1 se llame 'readADST_surface' 
        # o cambiá el nombre aquí abajo)
        df = readADST_surface_v6(root_fpath) 
        
        if df.empty:
            return f"INFO: Archivo vacío o sin datos UMD. Saltando: {filename}"

        # 2. Extraer metadatos del nombre de archivo
        try:
            parts = filename.split('_')
            df["model_mc"] = parts[0]
            df["e_min_mc"] = float(parts[1]) / 10.0
            df["e_max_mc"] = float(parts[2]) / 10.0
            df["primary_name_mc"] = parts[3]
            run_part = parts[-1].replace('.root', '')
            df["run_number"] = int(run_part.replace('Run', ''))
        except Exception as e_parse:
            print(f"  Advertencia: No se pudo parsear metadata en {filename}: {e_parse}")

        # 3. Guardar en Parquet
        df.to_parquet(
            output_path,
            compression="snappy",
            index=False
        )
        
        # 4. Liberar memoria
        del df
        
        end_file_time = time.time()
        elapsed = end_file_time - start_file_time
        return f"✔ [Éxito]: {filename} -> {output_filename} ({elapsed:.2f}s)"

        # ----- FIN DEL TRABAJO -----

    except Exception as e:
        # Si algo falla, retornamos el string de error
        return f"❌ [ERROR] en {filename}: {e}\n{traceback.format_exc()}"

# Prueba procesamiento con un ADST

In [29]:
print("--- INICIANDO PROCESO SERIAL (MODO DE PRUEBA: 1 ARCHIVO) ---")
start_total_time = time.time()

# --- 1. Configuración de Rutas ---
base_path = "/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/helium"
output_dir = "/home/lsilva/Github/Prueba_ADST_Alexey/parquet/"

os.makedirs(output_dir, exist_ok=True)
print(f"Buscando archivos en: {base_path}")
print(f"Los archivos .parquet se guardarán en: {output_dir}")

# --- 2. Encontrar todos los archivos ---
all_root_files = glob.glob(os.path.join(base_path, "*.root"))
all_root_files.sort()

if not all_root_files:
    print(f"¡Error! No se encontraron archivos .root en: {base_path}")
else:
    print(f"Encontrados {len(all_root_files)} archivos .root para procesar.")

# --- 3. Bucle de Procesamiento Serial ---
exitos = 0
errores = 0

for i, root_fpath in enumerate(all_root_files):
    
    filename = os.path.basename(root_fpath)
    output_filename = filename.replace(".root", ".parquet")
    output_path = os.path.join(output_dir, output_filename)
    
    print(f"\n--- Procesando archivo {i+1}/{len(all_root_files)} ---")
    print(f"► {filename}")
    
    # Evita reprocesar archivos que ya existen
    if os.path.exists(output_path):
        print(f"INFO: El archivo ya existe, saltando: {output_filename}")
        continue

    try:
        # ----- INICIO DEL TRABAJO -----
        start_file_time = time.time()
        
        # 1. Leer el .root (usa la Celda 2.1)
        df = readADST_surface_v6(root_fpath)
        
        if df.empty:
            print(f"INFO: Archivo vacío o sin datos UMD. Saltando.")
            continue

        # 2. Extraer metadatos del nombre de archivo
        try:
            parts = filename.split('_')
            df["model_mc"] = parts[0]
            df["e_min_mc"] = float(parts[1]) / 10.0
            df["e_max_mc"] = float(parts[2]) / 10.0
            df["primary_name_mc"] = parts[3]
            run_part = parts[-1].replace('.root', '')
            df["run_number"] = int(run_part.replace('Run', ''))
        except Exception as e_parse:
            print(f"  Advertencia: No se pudo parsear metadata: {e_parse}")

        # 3. Guardar en Parquet
        df.to_parquet(
            output_path,
            compression="snappy",
            index=False
        )
        
        # 4. Liberar memoria
        del df
        
        end_file_time = time.time()
        print(f"✔ [Éxito]: Guardado como {output_filename}")
        print(f"Tiempo de este archivo: {end_file_time - start_file_time:.2f}s")
        exitos += 1
        
        # ----- FIN DEL TRABAJO -----
        
        # ❗️ LÍNEA DE PRUEBA AÑADIDA ❗️
        print("\n--- ¡PRUEBA DETENIDA! Saliendo del bucle después de 1 archivo. ---")
        break  # Esto detiene el 'for' después de la primera pasada exitosa
               # ELIMINÁ ESTA LÍNEA para procesar todos los archivos.

    except Exception as e:
        print(f"❌ [ERROR] en {filename}: {e}")
        print(traceback.format_exc())
        errores += 1

# --- 4. Resumen Final ---
end_total_time = time.time()
print("\n\n--- Proceso Completado ---")
print(f"Tiempo total: {(end_total_time - start_total_time) / 60:.2f} minutos")
print(f"Total de archivos procesados: {exitos + errores}")
print(f"Éxitos: {exitos}")
print(f"Errores: {errores}")
print(f"¡Listo! Tu archivo .parquet de prueba está en: {output_dir}")

--- INICIANDO PROCESO SERIAL (MODO DE PRUEBA: 1 ARCHIVO) ---
Buscando archivos en: /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/helium
Los archivos .parquet se guardarán en: /home/lsilva/Github/Prueba_ADST_Alexey/parquet/
Encontrados 20 archivos .root para procesar.

--- Procesando archivo 1/20 ---
► SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run010.root
Iniciando lectura de: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run010.root
... procesados 500 eventos.
... procesados 1000 eventos.
Lectura completa. Total de eventos leídos: 1223
Total de 'MÓDULOS' (filas) extraídos: 81909
✔ [Éxito]: Guardado como SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run010.parquet
Tiempo de este archivo: 370.57s

--- ¡PRUEBA DETENIDA! Saliendo del bucle después de 1 archivo. ---


--- Proceso Completado ---
Tiempo total: 6.18 minutos
Total de archivos procesados: 1
Éxitos: 1
Errores: 0
¡Listo! Tu archivo .parquet de prueba está en:

  input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/helium/SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run010.root


In [32]:
# --- Cargar tu nuevo DataFrame (1 fila por MÓDULO) ---
parquet_path = "/home/lsilva/Github/Prueba_ADST_Alexey/parquet/SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run010.parquet"
df_new = pd.read_parquet(parquet_path)

print(f"DataFrame 'df_new' cargado con {len(df_new)} filas (MÓDULOS).")

# --- 1. Inspeccionar las nuevas columnas ---
print("\n--- Head (con nuevas columnas) ---")

print(df_new.head(10)) # Muestro 10 filas para ver el cambio de módulo

# --- 2. ¡La verificación CLAVE! ---
# Ver qué estados de módulo capturaste
print("\n--- Conteo de 'module_status' ---")
print(df_new['module_status'].value_counts())

DataFrame 'df_new' cargado con 81909 filas (MÓDULOS).

--- Head (con nuevas columnas) ---
                                            event_id    logE_MC   theta_MC  \
0  Library_Prague_Santos_Yushkov:Run_10000:Shower...  17.557002  10.497525   
1  Library_Prague_Santos_Yushkov:Run_10000:Shower...  17.557002  10.497525   
2  Library_Prague_Santos_Yushkov:Run_10000:Shower...  17.557002  10.497525   
3  Library_Prague_Santos_Yushkov:Run_10000:Shower...  17.557002  10.497525   
4  Library_Prague_Santos_Yushkov:Run_10000:Shower...  17.557002  10.497525   
5  Library_Prague_Santos_Yushkov:Run_10000:Shower...  17.557002  10.497525   
6  Library_Prague_Santos_Yushkov:Run_10000:Shower...  17.557002  10.497525   
7  Library_Prague_Santos_Yushkov:Run_10000:Shower...  17.557002  10.497525   
8  Library_Prague_Santos_Yushkov:Run_10000:Shower...  17.557002  10.497525   
9  Library_Prague_Santos_Yushkov:Run_10000:Shower...  17.557002  10.497525   

       phi_MC primary   logE_REC  theta_REC     phi

In [34]:
print(df_new['counterId'].unique())

[ 90000  90001  90002  90003  90004  90005  90006  90007  90008  90009
  90010  90011 104009 104010 104021 104022 104023 104040 104041 104032
 104033 104054 104055 104056 104075 104215 104216 104074 104214 104218
 104001 104002 104003 104004 104005 104007 104008 104011 104012 104018
 104019 104013 104025 104026 104027 104044 104045 104047 104067 104015
 104016 104017 104029 104030 104031 104052 104024 104043 104014 104028
 104049 104053 104073 104212 104213 104217 104048 104051 104006 104034
 104035 104057 104058 104046 104020 104036 104039 104038 104037 104060
 104061 104042 104059 104163 104064 104161 104050 104096 104079 104160
 104066 104068 104106 104072 104097 104120 104188 104062 104158 104132
 104151 104095 104150 104114 104082 104093 104116 104069 104130 104063
 104070 104078 104081 104092 104129 104065 104085 104119 104147 104071
 104125 104193 104099 104136 104137 104076 104098 104091 104206 104115
 104126 104122 104077 104133 104101]


Los de antes (AHORA ANDA JOYA VAMOOOOS):

[4022 4023 4041 4021 4009 4010 4040 4216 4055 4215 4033 4032 4054 4056
 4075 4218 4214 4074 4003 4002 4001 4004 4011 4008 4019 4007 4018 4012
 4005 4025 4026 4045 4027 4047 4044 4013 4067 4031 4016 4015 4030 4029
 4052 4017 4024 4043 4014 4028 4049 4053 4212 4213 4073 4217 4048 4051
 4006 4034 4035 4058 4057 4046 4039 4020 4036 4038 4037 4061 4060 4042
 4059 4163 4064 4050 4096 4066 4097 4072 4188 4158 4151 4095 4116 4069
 4063 4130 4078 4081 4125 4071 4193 4106 4079 4206 4132 4085 4129 4070
 4076]

# Paralelizacion (4 instancias) -> Sibyl-Helio-17.5-18eV

In [6]:
# -----------------------------------------------------------------
# FUNCIÓN "TRABAJADORA"
# Esta función contiene toda la lógica que antes estaba en tu bucle.
# Se ejecutará en un proceso separado para CADA archivo.
# -----------------------------------------------------------------
def process_file_wrapper(root_fpath):
    
    # 1. Definir rutas
    output_dir = "/home/lsilva/Github/ADST_Alexey_por_module/parquet/" # Ruta de salida
    filename = os.path.basename(root_fpath)
    output_filename = filename.replace(".root", ".parquet")
    output_path = os.path.join(output_dir, output_filename)
    
    # 2. Evitar reprocesar
    if os.path.exists(output_path):
        return f"INFO: El archivo ya existe, saltando: {output_filename}"

    # 3. Imprimir estado
    # (En paralelo, los prints pueden salir desordenados, es normal)
    print(f"► [Iniciando]: {filename}")
    
    try:
        # ----- INICIO DEL TRABAJO -----
        start_file_time = time.time()
        
        # 1. Leer el .root (¡Llama a la función de la Celda 2.1!)
        # (Asegurate de haber ejecutado tu Celda 2.1 con la
        # función readADST_surface() antes de correr esta celda)
        df = readADST_surface_v6(root_fpath)
        
        if df.empty:
            return f"INFO: Archivo vacío o sin datos UMD. Saltando: {filename}"

        # 2. Extraer metadatos del nombre de archivo
        try:
            parts = filename.split('_')
            df["model_mc"] = parts[0]
            df["e_min_mc"] = float(parts[1]) / 10.0
            df["e_max_mc"] = float(parts[2]) / 10.0
            df["primary_name_mc"] = parts[3]
            run_part = parts[-1].replace('.root', '')
            df["run_number"] = int(run_part.replace('Run', ''))
        except Exception as e_parse:
            print(f"  Advertencia: No se pudo parsear metadata en {filename}: {e_parse}")

        # 3. Guardar en Parquet
        df.to_parquet(
            output_path,
            compression="snappy",
            index=False
        )
        
        # 4. Liberar memoria
        del df
        
        end_file_time = time.time()
        elapsed = end_file_time - start_file_time
        return f"✔ [Éxito]: {filename} -> {output_filename} ({elapsed:.2f}s)"

        # ----- FIN DEL TRABAJO -----

    except Exception as e:
        # Si algo falla, retornamos el string de error
        return f"❌ [ERROR] en {filename}: {e}\n{traceback.format_exc()}"

In [7]:
# -----------------------------------------------------------------
# SCRIPT PRINCIPAL
# -----------------------------------------------------------------

print("--- INICIANDO PROCESO PARALELO (4 Workers) ---")
start_total_time = time.time()

# --- 1. Configuración de Rutas ---
base_path = "/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/helium"
output_dir = "/home/lsilva/Github/ADST_Alexey_por_module/parquet/"

os.makedirs(output_dir, exist_ok=True)
print(f"Buscando archivos en: {base_path}")
print(f"Los archivos .parquet se guardarán en: {output_dir}")

# --- 2. Encontrar todos los archivos ---
all_root_files = glob.glob(os.path.join(base_path, "*.root"))
all_root_files.sort()

if not all_root_files:
    print(f"¡Error! No se encontraron archivos .root en: {base_path}")
else:
    print(f"Encontrados {len(all_root_files)} archivos .root para procesar.")

    # --- 3. Bucle de Procesamiento Paralelo ---
    n_workers = 4  # Ajustar como sea necesario
    print(f"Iniciando Pool con {n_workers} trabajadores...")


    with Pool(processes=n_workers) as pool:
        
        # pool.map() distribuye la lista 'all_root_files'
        # entre los 4 trabajadores y aplica la función 'process_file_wrapper'
        # 'results' será una lista con los strings de "Éxito" o "ERROR"
        results = pool.map(process_file_wrapper, all_root_files)
    
    print("\n\n--- Proceso Paralelo Completado ---")

    # --- 4. Resumen Final ---
    exitos = 0
    errores = 0
    
    # Imprimimos todos los mensajes de resultado
    for res in results:
        print(res)
        if "✔ [Éxito]" in res:
            exitos += 1
        elif "❌ [ERROR]" in res:
            errores += 1

    end_total_time = time.time()
    print("\n--- Resumen de la Tanda ---")
    print(f"Tiempo total: {(end_total_time - start_total_time) / 60:.2f} minutos")
    print(f"Total de archivos: {len(all_root_files)}")
    print(f"Éxitos: {exitos}")
    print(f"Errores: {errores}")
    print(f"¡Listo! Tus archivos .parquet están en: {output_dir}")

--- INICIANDO PROCESO PARALELO (4 Workers) ---
Buscando archivos en: /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/helium
Los archivos .parquet se guardarán en: /home/lsilva/Github/ADST_Alexey_por_module/parquet/
Encontrados 20 archivos .root para procesar.
Iniciando Pool con 4 trabajadores...
► [Iniciando]: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run012.root► [Iniciando]: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run031.root► [Iniciando]: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run010.root► [Iniciando]: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run014.root



Iniciando lectura de: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run031.rootIniciando lectura de: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run012.rootIniciando lectura de: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run014.root

Iniciando lectura de: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run010.

  input file   input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/helium/SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run010.root/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/helium/SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run014.root

  input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/helium/SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run031.root
  input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/helium/SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run012.root
Warning in <TStreamerInfo::BuildCheck>: 
   The StreamerInfo of class Detector read from file /srv/data/Malargue/icrc2025/test7/Warning in <TStreamerInfo::BuildCheck>: 
   The StreamerInfo of class Detector read from file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSd

# Paralelizacion -> Sibyl-Hierro-17.5-18eV

In [9]:
# -----------------------------------------------------------------
# SCRIPT PRINCIPAL
# -----------------------------------------------------------------

print("--- INICIANDO PROCESO PARALELO (4 Workers) ---")
start_total_time = time.time()

# --- 1. Configuración de Rutas ---
base_path = "/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/iron"
output_dir = "/home/lsilva/Github/ADST_Alexey_module/parquet_hierro/"

os.makedirs(output_dir, exist_ok=True)
print(f"Buscando archivos en: {base_path}")
print(f"Los archivos .parquet se guardarán en: {output_dir}")

# --- 2. Encontrar todos los archivos ---
all_root_files = glob.glob(os.path.join(base_path, "*.root"))
all_root_files.sort()

if not all_root_files:
    print(f"¡Error! No se encontraron archivos .root en: {base_path}")
else:
    print(f"Encontrados {len(all_root_files)} archivos .root para procesar.")

    # --- 3. Bucle de Procesamiento Paralelo ---
    n_workers = 4  # Ajustar como sea necesario
    print(f"Iniciando Pool con {n_workers} trabajadores...")


    with Pool(processes=n_workers) as pool:
        
        # pool.map() distribuye la lista 'all_root_files'
        # entre los 4 trabajadores y aplica la función 'process_file_wrapper'
        # 'results' será una lista con los strings de "Éxito" o "ERROR"
        results = pool.map(process_file_wrapper, all_root_files)
    
    print("\n\n--- Proceso Paralelo Completado ---")

    # --- 4. Resumen Final ---
    exitos = 0
    errores = 0
    
    # Imprimimos todos los mensajes de resultado
    for res in results:
        print(res)
        if "✔ [Éxito]" in res:
            exitos += 1
        elif "❌ [ERROR]" in res:
            errores += 1

    end_total_time = time.time()
    print("\n--- Resumen de la Tanda ---")
    print(f"Tiempo total: {(end_total_time - start_total_time) / 60:.2f} minutos")
    print(f"Total de archivos: {len(all_root_files)}")
    print(f"Éxitos: {exitos}")
    print(f"Errores: {errores}")
    print(f"¡Listo! Tus archivos .parquet están en: {output_dir}")

--- INICIANDO PROCESO PARALELO (4 Workers) ---
Buscando archivos en: /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/iron
Los archivos .parquet se guardarán en: /home/lsilva/Github/ADST_Alexey_module/parquet_hierro/
Encontrados 20 archivos .root para procesar.
Iniciando Pool con 4 trabajadores...
► [Iniciando]: SIB23e_175_180_iron_MdSdInfill_CORSIKA78010_FLUKA_Run014.root► [Iniciando]: SIB23e_175_180_iron_MdSdInfill_CORSIKA78010_FLUKA_Run012.root► [Iniciando]: SIB23e_175_180_iron_MdSdInfill_CORSIKA78010_FLUKA_Run031.root► [Iniciando]: SIB23e_175_180_iron_MdSdInfill_CORSIKA78010_FLUKA_Run010.root



Iniciando lectura de: SIB23e_175_180_iron_MdSdInfill_CORSIKA78010_FLUKA_Run031.rootIniciando lectura de: SIB23e_175_180_iron_MdSdInfill_CORSIKA78010_FLUKA_Run014.rootIniciando lectura de: SIB23e_175_180_iron_MdSdInfill_CORSIKA78010_FLUKA_Run012.rootIniciando lectura de: SIB23e_175_180_iron_MdSdInfill_CORSIKA78010_FLUKA_Run010.root



... proce

  input file   input file   input file   input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/iron/SIB23e_175_180_iron_MdSdInfill_CORSIKA78010_FLUKA_Run010.root/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/iron/SIB23e_175_180_iron_MdSdInfill_CORSIKA78010_FLUKA_Run014.root/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/iron/SIB23e_175_180_iron_MdSdInfill_CORSIKA78010_FLUKA_Run012.root
/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/iron/SIB23e_175_180_iron_MdSdInfill_CORSIKA78010_FLUKA_Run031.root


Warning in <TStreamerInfo::BuildCheck>: 
   The StreamerInfo of class Detector read from file /srv/data/Malargue/icrc2025/test7/Warning in <TStreamerInfo::BuildCheck>: 
   The StreamerInfo of class Detector read from file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78

# Paralelizacion -> EPOS-Helio-18-18.5eV

In [10]:
# -----------------------------------------------------------------
# SCRIPT PRINCIPAL
# -----------------------------------------------------------------

print("--- INICIANDO PROCESO PARALELO (4 Workers) ---")
start_total_time = time.time()

# --- 1. Configuración de Rutas ---
base_path = "/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/EPOSLHC_R/18.0_18.5/helium"
output_dir = "/home/lsilva/Github/ADST_Alexey_module/parquet_epos_helio_18/"

os.makedirs(output_dir, exist_ok=True)
print(f"Buscando archivos en: {base_path}")
print(f"Los archivos .parquet se guardarán en: {output_dir}")

# --- 2. Encontrar todos los archivos ---
all_root_files = glob.glob(os.path.join(base_path, "*.root"))
all_root_files.sort()

if not all_root_files:
    print(f"¡Error! No se encontraron archivos .root en: {base_path}")
else:
    print(f"Encontrados {len(all_root_files)} archivos .root para procesar.")

    # --- 3. Bucle de Procesamiento Paralelo ---
    n_workers = 4  # Ajustar como sea necesario
    print(f"Iniciando Pool con {n_workers} trabajadores...")


    with Pool(processes=n_workers) as pool:
        
        # pool.map() distribuye la lista 'all_root_files'
        # entre los 4 trabajadores y aplica la función 'process_file_wrapper'
        # 'results' será una lista con los strings de "Éxito" o "ERROR"
        results = pool.map(process_file_wrapper, all_root_files)
    
    print("\n\n--- Proceso Paralelo Completado ---")

    # --- 4. Resumen Final ---
    exitos = 0
    errores = 0
    
    # Imprimimos todos los mensajes de resultado
    for res in results:
        print(res)
        if "✔ [Éxito]" in res:
            exitos += 1
        elif "❌ [ERROR]" in res:
            errores += 1

    end_total_time = time.time()
    print("\n--- Resumen de la Tanda ---")
    print(f"Tiempo total: {(end_total_time - start_total_time) / 60:.2f} minutos")
    print(f"Total de archivos: {len(all_root_files)}")
    print(f"Éxitos: {exitos}")
    print(f"Errores: {errores}")
    print(f"¡Listo! Tus archivos .parquet están en: {output_dir}")

--- INICIANDO PROCESO PARALELO (4 Workers) ---
Buscando archivos en: /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/EPOSLHC_R/18.0_18.5/helium
Los archivos .parquet se guardarán en: /home/lsilva/Github/ADST_Alexey_module/parquet_epos_helio_18/
Encontrados 20 archivos .root para procesar.
Iniciando Pool con 4 trabajadores...
► [Iniciando]: EPOSLHC_R_180_185_helium_MdSdInfill_CORSIKA78010_FLUKA_Run031.root► [Iniciando]: EPOSLHC_R_180_185_helium_MdSdInfill_CORSIKA78010_FLUKA_Run010.root► [Iniciando]: EPOSLHC_R_180_185_helium_MdSdInfill_CORSIKA78010_FLUKA_Run014.root► [Iniciando]: EPOSLHC_R_180_185_helium_MdSdInfill_CORSIKA78010_FLUKA_Run012.root



Iniciando lectura de: EPOSLHC_R_180_185_helium_MdSdInfill_CORSIKA78010_FLUKA_Run012.rootIniciando lectura de: EPOSLHC_R_180_185_helium_MdSdInfill_CORSIKA78010_FLUKA_Run031.rootIniciando lectura de: EPOSLHC_R_180_185_helium_MdSdInfill_CORSIKA78010_FLUKA_Run014.rootIniciando lectura de: EPOSLHC_R_180_185_helium_Md

  input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/EPOSLHC_R/18.0_18.5/helium/EPOSLHC_R_180_185_helium_MdSdInfill_CORSIKA78010_FLUKA_Run014.root
  input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/EPOSLHC_R/18.0_18.5/helium/EPOSLHC_R_180_185_helium_MdSdInfill_CORSIKA78010_FLUKA_Run010.root
  input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/EPOSLHC_R/18.0_18.5/helium/EPOSLHC_R_180_185_helium_MdSdInfill_CORSIKA78010_FLUKA_Run012.root
  input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/EPOSLHC_R/18.0_18.5/helium/EPOSLHC_R_180_185_helium_MdSdInfill_CORSIKA78010_FLUKA_Run031.root
Warning in <TStreamerInfo::BuildCheck>: 
   The StreamerInfo of class Detector read from file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/EPOSLHC_R/18.0_18.5/helium/EPOSLHC_R_180_185_helium_MdSdInfill_CORSIKA78010_FL

# Paralelizacion -> Sibyl-Oxigeno-17.5-18eV

In [11]:
# -----------------------------------------------------------------
# SCRIPT PRINCIPAL
# -----------------------------------------------------------------

print("--- INICIANDO PROCESO PARALELO (4 Workers) ---")
start_total_time = time.time()

# --- 1. Configuración de Rutas ---
base_path = "/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/oxygen"
output_dir = "/home/lsilva/Github/ADST_Alexey_module/parquet_sib_oxigeno_17/"

os.makedirs(output_dir, exist_ok=True)
print(f"Buscando archivos en: {base_path}")
print(f"Los archivos .parquet se guardarán en: {output_dir}")

# --- 2. Encontrar todos los archivos ---
all_root_files = glob.glob(os.path.join(base_path, "*.root"))
all_root_files.sort()

if not all_root_files:
    print(f"¡Error! No se encontraron archivos .root en: {base_path}")
else:
    print(f"Encontrados {len(all_root_files)} archivos .root para procesar.")

    # --- 3. Bucle de Procesamiento Paralelo ---
    n_workers = 4  # Ajustar como sea necesario
    print(f"Iniciando Pool con {n_workers} trabajadores...")


    with Pool(processes=n_workers) as pool:
        
        # pool.map() distribuye la lista 'all_root_files'
        # entre los 4 trabajadores y aplica la función 'process_file_wrapper'
        # 'results' será una lista con los strings de "Éxito" o "ERROR"
        results = pool.map(process_file_wrapper, all_root_files)
    
    print("\n\n--- Proceso Paralelo Completado ---")

    # --- 4. Resumen Final ---
    exitos = 0
    errores = 0
    
    # Imprimimos todos los mensajes de resultado
    for res in results:
        print(res)
        if "✔ [Éxito]" in res:
            exitos += 1
        elif "❌ [ERROR]" in res:
            errores += 1

    end_total_time = time.time()
    print("\n--- Resumen de la Tanda ---")
    print(f"Tiempo total: {(end_total_time - start_total_time) / 60:.2f} minutos")
    print(f"Total de archivos: {len(all_root_files)}")
    print(f"Éxitos: {exitos}")
    print(f"Errores: {errores}")
    print(f"¡Listo! Tus archivos .parquet están en: {output_dir}")

--- INICIANDO PROCESO PARALELO (4 Workers) ---
Buscando archivos en: /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/oxygen
Los archivos .parquet se guardarán en: /home/lsilva/Github/ADST_Alexey_module/parquet_sib_oxigeno_17/
Encontrados 20 archivos .root para procesar.
Iniciando Pool con 4 trabajadores...
► [Iniciando]: SIB23e_175_180_oxygen_MdSdInfill_CORSIKA78010_FLUKA_Run010.root► [Iniciando]: SIB23e_175_180_oxygen_MdSdInfill_CORSIKA78010_FLUKA_Run012.root► [Iniciando]: SIB23e_175_180_oxygen_MdSdInfill_CORSIKA78010_FLUKA_Run014.root► [Iniciando]: SIB23e_175_180_oxygen_MdSdInfill_CORSIKA78010_FLUKA_Run031.root



Iniciando lectura de: SIB23e_175_180_oxygen_MdSdInfill_CORSIKA78010_FLUKA_Run010.rootIniciando lectura de: SIB23e_175_180_oxygen_MdSdInfill_CORSIKA78010_FLUKA_Run012.rootIniciando lectura de: SIB23e_175_180_oxygen_MdSdInfill_CORSIKA78010_FLUKA_Run031.rootIniciando lectura de: SIB23e_175_180_oxygen_MdSdInfill_CORSIKA78010_FLUK

  input file   input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/oxygen/SIB23e_175_180_oxygen_MdSdInfill_CORSIKA78010_FLUKA_Run031.root/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/oxygen/SIB23e_175_180_oxygen_MdSdInfill_CORSIKA78010_FLUKA_Run012.root

  input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/oxygen/SIB23e_175_180_oxygen_MdSdInfill_CORSIKA78010_FLUKA_Run014.root
  input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/oxygen/SIB23e_175_180_oxygen_MdSdInfill_CORSIKA78010_FLUKA_Run010.root
Warning in <TStreamerInfo::BuildCheck>: 
   The StreamerInfo of class Detector read from file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/oxygen/SIB23e_175_180_oxygen_MdSdInfill_CORSIKA78010_FLUKA_Run010.root
   has the sam

# Paralelizacion -> Sibyl-Proton-17.5-18eV

In [20]:
# -----------------------------------------------------------------
# SCRIPT PRINCIPAL
# -----------------------------------------------------------------

print("--- INICIANDO PROCESO PARALELO (4 Workers) ---")
start_total_time = time.time()

# --- 1. Configuración de Rutas ---
base_path = "/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/proton"
output_dir = "/home/lsilva/Github/ADST_Alexey_module/parquet_sib_proton_17/"

os.makedirs(output_dir, exist_ok=True)
print(f"Buscando archivos en: {base_path}")
print(f"Los archivos .parquet se guardarán en: {output_dir}")

# --- 2. Encontrar todos los archivos ---
all_root_files = glob.glob(os.path.join(base_path, "*.root"))
all_root_files.sort()

if not all_root_files:
    print(f"¡Error! No se encontraron archivos .root en: {base_path}")
else:
    print(f"Encontrados {len(all_root_files)} archivos .root para procesar.")
    
    # --- 3. Bucle de Procesamiento Paralelo ---
    n_workers = 4  # Ajustar como sea necesario
    print(f"Iniciando Pool con {n_workers} trabajadores...")

    # CREAMOS LA FUNCIÓN "PARCIAL" 
    # "Congelamos" el argumento 'output_dir' de nuestra función wrapper
    process_func = partial(process_file_wrapper, output_dir=output_dir)

    with Pool(processes=n_workers) as pool:
        
        # pool.map() distribuye la lista 'all_root_files'
        # entre los 4 trabajadores y aplica la función 'process_file_wrapper'
        # results' será una lista con los strings de "Éxito" o "ERROR"
        results = pool.map(process_func, all_root_files)
    
    print("\n\n--- Proceso Paralelo Completado ---")

    # --- 4. Resumen Final ---
    exitos = 0
    errores = 0
    
    # Imprimimos todos los mensajes de resultado
    for res in results:
        print(res)
        if "✔ [Éxito]" in res:
            exitos += 1
        elif "❌ [ERROR]" in res:
            errores += 1

    end_total_time = time.time()
    print("\n--- Resumen de la Tanda ---")
    print(f"Tiempo total: {(end_total_time - start_total_time) / 60:.2f} minutos")
    print(f"Total de archivos: {len(all_root_files)}")
    print(f"Éxitos: {exitos}")
    print(f"Errores: {errores}")
    print(f"¡Listo! Tus archivos .parquet están en: {output_dir}")

--- INICIANDO PROCESO PARALELO (4 Workers) ---
Buscando archivos en: /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/proton
Los archivos .parquet se guardarán en: /home/lsilva/Github/ADST_Alexey_module/parquet_sib_proton_17/
Encontrados 20 archivos .root para procesar.
Iniciando Pool con 4 trabajadores...
► [Iniciando]: SIB23e_175_180_proton_MdSdInfill_CORSIKA78010_FLUKA_Run014.root► [Iniciando]: SIB23e_175_180_proton_MdSdInfill_CORSIKA78010_FLUKA_Run031.root► [Iniciando]: SIB23e_175_180_proton_MdSdInfill_CORSIKA78010_FLUKA_Run010.root► [Iniciando]: SIB23e_175_180_proton_MdSdInfill_CORSIKA78010_FLUKA_Run012.root



Iniciando lectura de: SIB23e_175_180_proton_MdSdInfill_CORSIKA78010_FLUKA_Run031.rootIniciando lectura de: SIB23e_175_180_proton_MdSdInfill_CORSIKA78010_FLUKA_Run010.rootIniciando lectura de: SIB23e_175_180_proton_MdSdInfill_CORSIKA78010_FLUKA_Run014.rootIniciando lectura de: SIB23e_175_180_proton_MdSdInfill_CORSIKA78010_FLUKA

  input file   input file   input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/proton/SIB23e_175_180_proton_MdSdInfill_CORSIKA78010_FLUKA_Run012.root/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/proton/SIB23e_175_180_proton_MdSdInfill_CORSIKA78010_FLUKA_Run010.root/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/proton/SIB23e_175_180_proton_MdSdInfill_CORSIKA78010_FLUKA_Run014.root


  input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/proton/SIB23e_175_180_proton_MdSdInfill_CORSIKA78010_FLUKA_Run031.root
Warning in <TStreamerInfo::BuildCheck>: 
   The StreamerInfo of class Detector read from file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/proton/SIB23e_175_180_proton_MdSdInfill_CORSIKA78010_FLUKA_Run010.root
   has the sam

# Paralelizacion -> QGS-Helium-17.5-18eV

In [21]:
# -----------------------------------------------------------------
# SCRIPT PRINCIPAL
# -----------------------------------------------------------------

print("--- INICIANDO PROCESO PARALELO (4 Workers) ---")
start_total_time = time.time()

# --- 1. Configuración de Rutas ---
base_path = "/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/QGSIII01/17.5_18.0/helium"
output_dir = "/home/lsilva/Github/ADST_Alexey_module/parquet_qgs_helium_17/"

os.makedirs(output_dir, exist_ok=True)
print(f"Buscando archivos en: {base_path}")
print(f"Los archivos .parquet se guardarán en: {output_dir}")

# --- 2. Encontrar todos los archivos ---
all_root_files = glob.glob(os.path.join(base_path, "*.root"))
all_root_files.sort()

if not all_root_files:
    print(f"¡Error! No se encontraron archivos .root en: {base_path}")
else:
    print(f"Encontrados {len(all_root_files)} archivos .root para procesar.")
    
    # --- 3. Bucle de Procesamiento Paralelo ---
    n_workers = 4  # Ajustar como sea necesario
    print(f"Iniciando Pool con {n_workers} trabajadores...")

    # CREAMOS LA FUNCIÓN "PARCIAL" 
    # "Congelamos" el argumento 'output_dir' de nuestra función wrapper
    process_func = partial(process_file_wrapper, output_dir=output_dir)

    with Pool(processes=n_workers) as pool:
        
        # pool.map() distribuye la lista 'all_root_files'
        # entre los 4 trabajadores y aplica la función 'process_file_wrapper'
        # results' será una lista con los strings de "Éxito" o "ERROR"
        results = pool.map(process_func, all_root_files)
    
    print("\n\n--- Proceso Paralelo Completado ---")

    # --- 4. Resumen Final ---
    exitos = 0
    errores = 0
    
    # Imprimimos todos los mensajes de resultado
    for res in results:
        print(res)
        if "✔ [Éxito]" in res:
            exitos += 1
        elif "❌ [ERROR]" in res:
            errores += 1

    end_total_time = time.time()
    print("\n--- Resumen de la Tanda ---")
    print(f"Tiempo total: {(end_total_time - start_total_time) / 60:.2f} minutos")
    print(f"Total de archivos: {len(all_root_files)}")
    print(f"Éxitos: {exitos}")
    print(f"Errores: {errores}")
    print(f"¡Listo! Tus archivos .parquet están en: {output_dir}")

--- INICIANDO PROCESO PARALELO (4 Workers) ---
Buscando archivos en: /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/QGSIII01/17.5_18.0/helium
Los archivos .parquet se guardarán en: /home/lsilva/Github/ADST_Alexey_module/parquet_qgs_helium_17/
Encontrados 20 archivos .root para procesar.
Iniciando Pool con 4 trabajadores...
► [Iniciando]: QGSIII01_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run010.root► [Iniciando]: QGSIII01_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run014.root► [Iniciando]: QGSIII01_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run012.root► [Iniciando]: QGSIII01_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run031.root



Iniciando lectura de: QGSIII01_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run010.rootIniciando lectura de: QGSIII01_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run014.rootIniciando lectura de: QGSIII01_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run012.rootIniciando lectura de: QGSIII01_175_180_helium_MdSdInfill_

  input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/QGSIII01/17.5_18.0/helium/QGSIII01_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run014.root
  input file   input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/QGSIII01/17.5_18.0/helium/QGSIII01_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run031.root/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/QGSIII01/17.5_18.0/helium/QGSIII01_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run010.root

  input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/QGSIII01/17.5_18.0/helium/QGSIII01_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run012.root
Warning in <TStreamerInfo::BuildCheck>: 
   The StreamerInfo of class Detector read from file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/QGSIII01/17.5_18.0/helium/QGSIII01_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run014